# MAADS-BML Training

This notebook prepares **bank transactions** dataset for MAADS-BML **multiclass severity** training.

This notebook:
1) Loads raw CSV
2) Normalizes column names
3) Auto-detects a date column and creates **`Date`** as the **first** column in **M/D/YYYY** format
4) Creates multiclass target **`severity_label`**:
- legit -> 0
- soft_fraud -> 1
- medium_fraud -> 2
- hard_fraud -> 3
5) Drops common leakage columns (`fraud_score`, `fraud_reason_codes`, etc.)
6) Encodes categoricals (one-hot for low-cardinality; drops high-cardinality ID-like columns)
7) Forces all features (except `Date`) to be numeric (MAADS requirement)
8) Creates a **clean holdout split** (never augmented)
9) Augments only the **training** split by oversampling `hard_fraud` to a target count
10) Exports two MAADS-ready files:
- `maads_train_aug.csv`  (for MAADS training with `trainingpercentage=80`)
- `maads_holdout.csv`    (for evaluation)

In [1]:
# Imports

import pandas as pd
import numpy as np
import os
from pathlib import Path


In [ ]:
# Load environment (.env)

from dotenv import load_dotenv

dotenv_path = Path(os.environ.get(
    "DOTENV_PATH",
    Path.cwd().parent / ".env"
))
if not dotenv_path.exists():
    raise FileNotFoundError(
        f'.env not found at: {dotenv_path}. '
        'Start Jupyter from repo root or set DOTENV_PATH.'
    )

load_dotenv(dotenv_path=dotenv_path, override=False)


In [ ]:
# Configuration (from .env)

def env_required(key: str) -> str:
    v = os.environ.get(key)
    if v is None or str(v).strip() == '':
        raise KeyError(f"Missing required .env key: {key}")
    return str(v).strip()

def env_int(key: str, default: int) -> int:
    v = os.environ.get(key)
    return int(v) if v is not None and str(v).strip() != '' else int(default)

def env_float(key: str, default: float) -> float:
    v = os.environ.get(key)
    return float(v) if v is not None and str(v).strip() != '' else float(default)

# Paths
RAW_PATH = Path(os.environ.get("RAW_PATH", "data/bank_transactions_data_labeled.csv")).expanduser().resolve()
MAADS_CSVUPLOADS = Path(env_required("MAADS_CSVUPLOADS")).expanduser().resolve()

# Data prep knobs
HOLDOUT_FRAC = env_float("HOLDOUT_FRAC", 0.30)
HARD_FRAUD_TARGET_COUNT = env_int("HARD_FRAUD_TARGET_COUNT", 200)
NOISE_STD_FRAC = env_float("NOISE_STD_FRAC", 0.01)
MAX_DUMMIES = env_int("MAX_DUMMIES", 20)

# MAADS-BML service connection
host = env_required("MAADS_HOST").rstrip("/")
trainingport = env_int("MAADS_TRAINING_PORT", 5595)  # default MAADS TrainingService port
MAADS_TRAIN_URL = f"{host}:{trainingport}/"

# Auth / metadata for MAADS service
MAADS_USERNAME = env_required("MAADS_USERNAME")
MAADS_PASSWORD = env_required("MAADS_PASSWORD")
MAADS_TOKEN = env_required("MAADS_TOKEN")       # maadstoken
MAADS_COMPANY = env_required("MAADS_COMPANY")
MAADS_EMAIL = env_required("MAADS_EMAIL")

# Training file and target
MAADS_TRAIN_FILENAME = os.environ.get("MAADS_TRAIN_FILENAME", "maads_train_aug_binary.csv").strip()
MAADS_DEPENDENT_VARIABLE = os.environ.get("MAADS_DEPENDENT_VARIABLE", "label").strip()

# Training behavior
MAADS_REMOVE_OUTLIERS = env_int("MAADS_REMOVE_OUTLIERS", 0)
MAADS_HAS_SEASONALITY = env_int("MAADS_HAS_SEASONALITY", 0)
MAADS_SUMMER = os.environ.get("MAADS_SUMMER", "6,7,8")
MAADS_WINTER = os.environ.get("MAADS_WINTER", "11,12,1,2")
MAADS_SHOULDER = os.environ.get("MAADS_SHOULDER", "3,4,5,9,10")
MAADS_TRAINING_PERCENTAGE = env_int("MAADS_TRAINING_PERCENTAGE", 80)
MAADS_SHUFFLE = env_int("MAADS_SHUFFLE", 1)
MAADS_DEEP_ANALYSIS = env_int("MAADS_DEEP_ANALYSIS", 1)
MAADS_TRAINING_TIMEOUT_SEC = env_int("MAADS_TRAINING_TIMEOUT_SEC", 3600)

# Client-side request timeout
HTTP_REQUEST_TIMEOUT_SEC = env_int("HTTP_REQUEST_TIMEOUT_SEC", 30)


In [3]:
# Load raw

df_raw = pd.read_csv(RAW_PATH)
print("Loaded:", RAW_PATH)
print("Shape:", df_raw.shape)
df_raw.head()


Loaded: data/bank_transactions_data_labeled.csv
Shape: (2512, 39)


,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,...,high_amount,unusual_mcc_for_customer,txn_velocity_5min_sim,high_frequency,freq_2plus,proxy,fraud_label,fraud_score,fraud_severity,fraud_reason_codes
0,TX001313,AC00001,47.79,2023-09-15 17:00:20,Debit,Denver,D000649,59.12.96.11,M034,Branch,...,False,False,1,False,False,False,legit,0.06,low,"['NEW_DEVICE', 'NEW_IP', 'NEW_COUNTRY']"
1,TX002017,AC00001,212.97,2023-11-14 16:56:34,Debit,Atlanta,D000492,45.241.13.208,M003,Online,...,False,False,3,True,True,False,legit,0.09,low,"['HIGH_RISK_MCC', 'HIGH_FREQUENCY_5MIN', 'FREQ..."
2,TX002121,AC00002,476.99,2023-01-10 16:00:32,Debit,San Diego,D000594,113.137.153.101,M022,Online,...,False,False,1,False,False,False,legit,0.09,low,"['HIGH_RISK_MCC', 'NEW_DEVICE', 'NEW_IP', 'NEW..."
3,TX000021,AC00002,59.32,2023-02-28 16:36:58,Debit,Los Angeles,D000152,116.44.12.250,M040,Branch,...,False,False,1,False,False,False,legit,0.06,low,"['NEW_DEVICE', 'NEW_IP', 'NEW_COUNTRY']"
4,TX001477,AC00002,12.62,2023-05-05 16:35:44,Debit,El Paso,D000475,93.160.83.196,M068,Branch,...,False,False,1,False,False,False,legit,0.04,low,"['MEDIUM_RISK_MCC', 'NEW_DEVICE', 'NEW_IP']"


In [4]:
# Normalize column names

df = df_raw.copy()

def normalize_cols(cols):
    cols = pd.Index(cols)
    return (
        cols.astype(str)
            .str.strip()
            .str.lower()
            .str.replace(r"\s+", "_", regex=True)
            .str.replace("-", "_")
            .str.replace(".", "_", regex=False)
    )

df.columns = normalize_cols(df.columns)
df.columns.tolist()


['transactionid',
 'accountid',
 'transactionamount',
 'transactiondate',
 'transactiontype',
 'location',
 'deviceid',
 'ip_address',
 'merchantid',
 'channel',
 'customerage',
 'customeroccupation',
 'transactionduration',
 'loginattempts',
 'accountbalance',
 'previoustransactiondate',
 'txn_hour',
 'is_night',
 'is_weekend',
 'mcc',
 'mcc_risk',
 'high_risk_mcc',
 'medium_risk_mcc',
 'country',
 'new_country',
 'new_device',
 'new_ip',
 'customer_tenure_days',
 'short_tenure',
 'high_amount',
 'unusual_mcc_for_customer',
 'txn_velocity_5min_sim',
 'high_frequency',
 'freq_2plus',
 'proxy',
 'fraud_label',
 'fraud_score',
 'fraud_severity',
 'fraud_reason_codes']

In [5]:
# Auto-detect date column & create MAADS Date

def pick_date_col(columns):
    preferred = [
        "transactiondate", "transaction_date", "txn_date", "date",
        "datetime", "timestamp", "created_at",
        "previoustransactiondate", "previous_transaction_date"
    ]
    cols = list(columns)
    for p in preferred:
        if p in cols:
            return p
    date_like = [c for c in cols if "date" in c]
    return date_like[0] if date_like else None

date_col = pick_date_col(df.columns)
print("Detected date column:", date_col)
if date_col is None:
    raise ValueError("No date-like column found. Please set date_col manually.")

dt = pd.to_datetime(df[date_col], errors="coerce")
bad_n = int(dt.isna().sum())
if bad_n:
    sample_bad = df.loc[dt.isna(), date_col].head(10).tolist()
    raise ValueError(f"Found {bad_n} invalid dates in '{date_col}'. Examples: {sample_bad}")

# MAADS wants M/D/YYYY
df.insert(0, "Date", dt.dt.strftime("%m/%d/%Y"))
df = df.drop(columns=[date_col])
df.head()


Detected date column: transactiondate


,Date,transactionid,accountid,transactionamount,transactiontype,location,deviceid,ip_address,merchantid,channel,...,high_amount,unusual_mcc_for_customer,txn_velocity_5min_sim,high_frequency,freq_2plus,proxy,fraud_label,fraud_score,fraud_severity,fraud_reason_codes
0,09/15/2023,TX001313,AC00001,47.79,Debit,Denver,D000649,59.12.96.11,M034,Branch,...,False,False,1,False,False,False,legit,0.06,low,"['NEW_DEVICE', 'NEW_IP', 'NEW_COUNTRY']"
1,11/14/2023,TX002017,AC00001,212.97,Debit,Atlanta,D000492,45.241.13.208,M003,Online,...,False,False,3,True,True,False,legit,0.09,low,"['HIGH_RISK_MCC', 'HIGH_FREQUENCY_5MIN', 'FREQ..."
2,01/10/2023,TX002121,AC00002,476.99,Debit,San Diego,D000594,113.137.153.101,M022,Online,...,False,False,1,False,False,False,legit,0.09,low,"['HIGH_RISK_MCC', 'NEW_DEVICE', 'NEW_IP', 'NEW..."
3,02/28/2023,TX000021,AC00002,59.32,Debit,Los Angeles,D000152,116.44.12.250,M040,Branch,...,False,False,1,False,False,False,legit,0.06,low,"['NEW_DEVICE', 'NEW_IP', 'NEW_COUNTRY']"
4,05/05/2023,TX001477,AC00002,12.62,Debit,El Paso,D000475,93.160.83.196,M068,Branch,...,False,False,1,False,False,False,legit,0.04,low,"['MEDIUM_RISK_MCC', 'NEW_DEVICE', 'NEW_IP']"


In [6]:
# Create multiclass target severity_label

if "fraud_label" not in df.columns:
    raise ValueError("Expected a 'fraud_label' column in the raw dataset.")

severity_map = {
    "legit": 0,
    "soft_fraud": 1,
    "medium_fraud": 2,
    "hard_fraud": 3,
}

df["fraud_label"] = df["fraud_label"].astype(str).str.lower().str.strip()
df["severity_label"] = df["fraud_label"].map(severity_map)

if df["severity_label"].isna().any():
    unknown = df.loc[df["severity_label"].isna(), "fraud_label"].value_counts().head(20)
    raise ValueError(f"Unknown fraud_label values found. Examples:\n{unknown}")

df["severity_label"] = df["severity_label"].astype(int)
print(df["severity_label"].value_counts().sort_index())


0    1963
1      75
2     448
3      26
Name: severity_label, dtype: int64


In [7]:
# Drop leakage / non-MAADS-friendly columns
# Keep fraud_label only for auditing; drop before encoding to avoid leakage.

DROP_ALWAYS = [
    "fraud_score",
    "fraud_reason_codes",
    "fraud_severity",
    "fraud_label",  # removing the string label as we use severity_label instead
]

to_drop = [c for c in DROP_ALWAYS if c in df.columns]
print("Dropping:", to_drop)
df = df.drop(columns=to_drop, errors="ignore")
df.shape


Dropping: ['fraud_score', 'fraud_reason_codes', 'fraud_severity', 'fraud_label']


(2512, 36)

In [8]:
# Convert booleans -> 0/1

bool_cols = [c for c in df.columns if df[c].dtype == bool]
print("Bool cols:", bool_cols)
for c in bool_cols:
    df[c] = df[c].astype(int)
    

Bool cols: ['is_night', 'is_weekend', 'high_risk_mcc', 'medium_risk_mcc', 'new_country', 'new_device', 'new_ip', 'short_tenure', 'high_amount', 'unusual_mcc_for_customer', 'high_frequency', 'freq_2plus', 'proxy']


In [9]:
# Encode categoricals safely

obj_cols = df.select_dtypes(include=["object"]).columns.tolist()
obj_cols = [c for c in obj_cols if c != "Date"]

id_like, low_card, high_card = [], [], []
for c in obj_cols:
    nun = df[c].nunique(dropna=True)
    name = c
    if any(k in name for k in ["id", "ip", "device", "account", "merchant", "transaction"]) and nun > MAX_DUMMIES:
        id_like.append(c)
    elif nun <= MAX_DUMMIES:
        low_card.append(c)
    else:
        high_card.append(c)

print("ID-like high-card drop:", id_like)
print("Low-card one-hot:", low_card)
print("High-card drop:", high_card)

df = df.drop(columns=id_like + high_card, errors="ignore")
if low_card:
    df = pd.get_dummies(df, columns=low_card, dummy_na=True, drop_first=False)

df.shape


ID-like high-card drop: ['transactionid', 'accountid', 'deviceid', 'ip_address', 'merchantid', 'previoustransactiondate']
Low-card one-hot: ['transactiontype', 'channel', 'customeroccupation', 'mcc_risk', 'country']
High-card drop: ['location']


(2512, 52)

In [10]:
# Replace Inf/NaN and force numeric dtypes (MAADS requirement)

df = df.replace([np.inf, -np.inf], np.nan)
df = df.fillna(0)

# Force every non-Date column to be numeric
for c in df.columns:
    if c == "Date":
        continue
    if df[c].dtype == bool:
        df[c] = df[c].astype(int)
    elif df[c].dtype == object:
        df[c] = pd.to_numeric(df[c], errors="raise")

# Cast all features to float (keeps 0/1 as 0.0/1.0 (MAADS requirement))
feature_cols = [c for c in df.columns if c != "Date"]
df[feature_cols] = df[feature_cols].astype(float)

df.dtypes.head(20)


Date                         object
transactionamount           float64
customerage                 float64
transactionduration         float64
loginattempts               float64
accountbalance              float64
txn_hour                    float64
is_night                    float64
is_weekend                  float64
mcc                         float64
high_risk_mcc               float64
medium_risk_mcc             float64
new_country                 float64
new_device                  float64
new_ip                      float64
customer_tenure_days        float64
short_tenure                float64
high_amount                 float64
unusual_mcc_for_customer    float64
txn_velocity_5min_sim       float64
dtype: object

In [11]:
# Final MAADS schema validation

assert df.columns[0] == "Date", "First column must be Date"
assert "severity_label" in df.columns, "Missing target: severity_label"
non_numeric = df.drop(columns=["Date"]).select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    raise ValueError(f"Non-numeric columns remain (MAADS will fail): {non_numeric}")
print("Shape:", df.shape)
print(df["severity_label"].value_counts().sort_index())


Shape: (2512, 52)
0.0    1963
1.0      75
2.0     448
3.0      26
Name: severity_label, dtype: int64


In [12]:
# Create holdout split (not augmented)

from sklearn.model_selection import train_test_split

train_df, holdout_df = train_test_split(
    df,
    test_size=HOLDOUT_FRAC,
    random_state=42,
    stratify=df["severity_label"],
)

print("Train shape:", train_df.shape)
print("Holdout shape:", holdout_df.shape)
print("Holdout class distribution:\n", holdout_df["severity_label"].value_counts().sort_index())


Train shape: (1758, 52)
Holdout shape: (754, 52)
Holdout class distribution:
 0.0    589
1.0     23
2.0    134
3.0      8
Name: severity_label, dtype: int64


In [13]:
# Augment only training split: oversample hard_fraud (class=3)

MIN_CLASS = 3

minor = train_df[train_df["severity_label"] == MIN_CLASS]
print("Hard fraud in train before:", len(minor))

if len(minor) == 0:
    raise ValueError("No hard_fraud rows in TRAIN split. Increase data or adjust split.")

if len(minor) >= HARD_FRAUD_TARGET_COUNT:
    train_aug = train_df.copy()
    print("Already enough hard_fraud; no augmentation performed.")
else:
    needed = HARD_FRAUD_TARGET_COUNT - len(minor)
    synth = minor.sample(n=needed, replace=True, random_state=42).copy()

    # Identify binary columns (only {0,1} values) among features
    feature_cols = [c for c in train_df.columns if c not in ["Date", "severity_label"]]
    binary_cols = [c for c in feature_cols if set(train_df[c].unique()).issubset({0.0, 1.0})]
    cont_cols = [c for c in feature_cols if c not in binary_cols]

    # Add tiny noise to continuous columns
    rng = np.random.default_rng(42)
    for c in cont_cols:
        std = float(train_df[c].std())
        if np.isnan(std) or std == 0.0:
            continue
        synth[c] = synth[c] + rng.normal(0.0, NOISE_STD_FRAC * std, size=len(synth))

    # Keep binary columns exactly 0/1
    for c in binary_cols:
        synth[c] = synth[c].round().clip(0, 1)

    train_aug = pd.concat([train_df, synth], ignore_index=True)

print("Hard fraud in train after:", int((train_aug["severity_label"] == 3).sum()))
print("Train augmented class distribution:\n", train_aug["severity_label"].value_counts().sort_index())


Hard fraud in train before: 18
Hard fraud in train after: 200
Train augmented class distribution:
 0.0    1374
1.0      52
2.0     314
3.0     200
Name: severity_label, dtype: int64


In [14]:
# Drop correlated features suggested by MAADS
# Configure via env var if needed:
#   DROP_FEATURES=new_country,transactionamount,high_amount,channel_ATM,mcc_risk_high,accountbalance

DROP_FEATURES_RAW = os.getenv(
    "DROP_FEATURES",
    "new_country,transactionamount,high_amount,channel_ATM,mcc_risk_high,accountbalance"
)
DROP_FEATURES = [c.strip() for c in DROP_FEATURES_RAW.split(",") if c.strip()]

def _drop_from(name: str):
    if name in globals():
        obj = globals()[name]
        if isinstance(obj, pd.DataFrame):
            existing = [c for c in DROP_FEATURES if c in obj.columns]
            missing  = [c for c in DROP_FEATURES if c not in obj.columns]
            print(f"[{name}] Dropping (existing):", existing)
            print(f"[{name}] Not present (ok):   ", missing)
            globals()[name] = obj.drop(columns=existing, errors="ignore")
            print(f"[{name}] Shape after drop:", globals()[name].shape)
            return existing, missing
    return [], DROP_FEATURES

# Apply to relevant dataframes used in export
existing_all = []
missing_all = None

for nm in ["df", "train_df", "holdout_df", "train_aug"]:
    ex, ms = _drop_from(nm)
    existing_all.extend(ex)
    missing_all = ms 

existing_all = sorted(set(existing_all))
print("\nSummary: dropped these correlated features across datasets:", existing_all)

# Persist feature columns for prediction alignment.
base_df = None
if "train_aug" in globals() and isinstance(globals()["train_aug"], pd.DataFrame):
    base_df = globals()["train_aug"]
elif "df" in globals() and isinstance(globals()["df"], pd.DataFrame):
    base_df = globals()["df"]
else:
    raise RuntimeError("No dataframe found to compute feature columns (expected train_aug or df).")

exclude = {"Date", "severity_label", "label"}  # targets are excluded
feature_cols_after = [c for c in base_df.columns if c not in exclude]

from pathlib import Path
out_dir = Path("maads_runs")
out_dir.mkdir(exist_ok=True)

(out_dir / "feature_columns.txt").write_text("\n".join(feature_cols_after) + "\n", encoding="utf-8")
print("Saved feature_columns.txt with", len(feature_cols_after), "features ->", out_dir / "feature_columns.txt")

# Copy feature list to MAADS_CSVUPLOADS if configured
try:
    if "MAADS_CSVUPLOADS" in globals() and MAADS_CSVUPLOADS:
        dst_dir = Path(MAADS_CSVUPLOADS)
        dst_dir.mkdir(parents=True, exist_ok=True)
        (dst_dir / "feature_columns.txt").write_text("\n".join(feature_cols_after) + "\n", encoding="utf-8")
        print("Also saved feature_columns.txt ->", dst_dir / "feature_columns.txt")
except Exception as e:
    print("Warning: could not write feature_columns.txt to MAADS_CSVUPLOADS:", e)


[df] Dropping (existing): ['new_country', 'transactionamount', 'high_amount', 'channel_ATM', 'mcc_risk_high', 'accountbalance']
[df] Not present (ok):    []
[df] Shape after drop: (2512, 46)
[train_df] Dropping (existing): ['new_country', 'transactionamount', 'high_amount', 'channel_ATM', 'mcc_risk_high', 'accountbalance']
[train_df] Not present (ok):    []
[train_df] Shape after drop: (1758, 46)
[holdout_df] Dropping (existing): ['new_country', 'transactionamount', 'high_amount', 'channel_ATM', 'mcc_risk_high', 'accountbalance']
[holdout_df] Not present (ok):    []
[holdout_df] Shape after drop: (754, 46)
[train_aug] Dropping (existing): ['new_country', 'transactionamount', 'high_amount', 'channel_ATM', 'mcc_risk_high', 'accountbalance']
[train_aug] Not present (ok):    []
[train_aug] Shape after drop: (1940, 46)

Summary: dropped these correlated features across datasets: ['accountbalance', 'channel_ATM', 'high_amount', 'mcc_risk_high', 'new_country', 'transactionamount']
Saved featu

In [15]:
# Export MAADS-ready files (multiclass + binary)

TRAIN_MULTI_OUT   = "maads_train_aug.csv"
HOLDOUT_MULTI_OUT = "maads_holdout.csv"
TRAIN_BIN_OUT     = "maads_train_aug_binary.csv"
HOLDOUT_BIN_OUT   = "maads_holdout_binary.csv"

# Destination folder
dst_dir = Path(MAADS_CSVUPLOADS)
dst_dir.mkdir(parents=True, exist_ok=True)
print("MAADS_CSVUPLOADS:", dst_dir.resolve())

# Save multiclass artifacts
(dst_dir / TRAIN_MULTI_OUT).write_bytes(pd.DataFrame(train_aug).to_csv(index=False).encode())
(dst_dir / HOLDOUT_MULTI_OUT).write_bytes(pd.DataFrame(holdout_df).to_csv(index=False).encode())

# Create binary artifacts (in-memory)
def to_binary(df_in: pd.DataFrame) -> pd.DataFrame:
    dfb = df_in.copy()
    if "severity_label" not in dfb.columns:
        raise ValueError("severity_label missing")
    dfb["label"] = (dfb["severity_label"] > 0).astype(int)
    dfb = dfb.drop(columns=["severity_label"])  # binary pipeline

    # MAADS guards
    if "Date" not in dfb.columns:
        raise ValueError("Date column missing")
    # Ensure Date is first column
    cols = ["Date"] + [c for c in dfb.columns if c != "Date"]
    dfb = dfb[cols]
    non_numeric = dfb.drop(columns=["Date"]).select_dtypes(exclude=["number"]).columns.tolist()
    if non_numeric:
        raise ValueError(f"Non-numeric columns remain (MAADS will fail): {non_numeric}")
    return dfb

train_bin   = to_binary(train_aug)
holdout_bin = to_binary(holdout_df)

(dst_dir / TRAIN_BIN_OUT).write_bytes(train_bin.to_csv(index=False).encode())
(dst_dir / HOLDOUT_BIN_OUT).write_bytes(holdout_bin.to_csv(index=False).encode())

print("Saved to MAADS_CSVUPLOADS:")
for f in [TRAIN_MULTI_OUT, HOLDOUT_MULTI_OUT, TRAIN_BIN_OUT, HOLDOUT_BIN_OUT]:
    p = dst_dir / f
    print(" -", p)
print("Binary label distribution (train):", train_bin["label"].value_counts().to_dict())
print("Binary label distribution (holdout):", holdout_bin["label"].value_counts().to_dict())


MAADS_CSVUPLOADS: /home/user/projects/tml-fraud/supervised/maadsbml/csvuploads
Saved to MAADS_CSVUPLOADS:
 - maadsbml/csvuploads/maads_train_aug.csv
 - maadsbml/csvuploads/maads_holdout.csv
 - maadsbml/csvuploads/maads_train_aug_binary.csv
 - maadsbml/csvuploads/maads_holdout_binary.csv
Binary label distribution (train): {0: 1374, 1: 566}
Binary label distribution (holdout): {0: 589, 1: 165}


# MAADS Binary Training

In [16]:
# Binary dataset verification
# This cell validates that the binary training file exists and is MAADS-compatible.

p = Path(MAADS_CSVUPLOADS) / TRAIN_BIN_OUT
if not p.exists():
    raise FileNotFoundError(f"{TRAIN_BIN_OUT} not found. Re-run the Export cell first.")

df_bin = pd.read_csv(p)

# MAADS checks
if df_bin.columns[0] != "Date":
    raise ValueError("MAADS expects the first column to be 'Date'.")
if "label" not in df_bin.columns:
    raise ValueError("Binary label column 'label' is missing.")

non_numeric = df_bin.drop(columns=["Date"]).select_dtypes(exclude=["number"]).columns.tolist()
if non_numeric:
    raise ValueError(f"Non-numeric columns remain (MAADS will fail): {non_numeric}")

print("Binary training file ok:", TRAIN_BIN_OUT)
print("Rows:", len(df_bin))
print("Label distribution:\n", df_bin["label"].value_counts())


Binary training file ok: maads_train_aug_binary.csv
Rows: 1940
Label distribution:
 0    1374
1     566
Name: label, dtype: int64


In [17]:
# Pre-Train Cleanup
# Clears MAADS temp/run folders before starting a new training run.
# Keeps trained networks by default.

import subprocess

# Container name running MAADS-BML
CONTAINER_NAME = globals().get('CONTAINER_NAME', 'maadsbml')

CLEAN_CMD = r'''
set -e
BASE=/maads/agentfilesdocker/dist/maadsweb
test -d "$BASE" || (echo "BASE not found: $BASE" && exit 1)

# Clean temp/run folders (safe)
rm -rf "$BASE/staging/"*        2>/dev/null || true
rm -rf "$BASE/networktemp/"*    2>/dev/null || true
rm -rf "$BASE/exception/"*      2>/dev/null || true

# Optional: uncomment if you want to clear previous reports/outliers too
# rm -rf "$BASE/outliers/"*     2>/dev/null || true
# rm -rf "$BASE/pdfreports/"*   2>/dev/null || true

echo "[cleanup] done"
'''

subprocess.check_call(["docker", "exec", "-i", CONTAINER_NAME, "sh", "-lc", CLEAN_CMD])
print(f"MAADS temp folders cleaned in container: {CONTAINER_NAME}")


[cleanup] done
MAADS temp folders cleaned in container: maadsbml


In [ ]:
# Run MAADS hypertraining

import json
import requests

params = {
    "hypertraining": 1,
    "mode": 0,
    "username": MAADS_USERNAME,
    "password": MAADS_PASSWORD,
    "maadstoken": MAADS_TOKEN,
    "company": MAADS_COMPANY,
    "email": MAADS_EMAIL,
    "filename": MAADS_TRAIN_FILENAME,
    "dependentvariable": MAADS_DEPENDENT_VARIABLE,
    "removeoutliers": int(MAADS_REMOVE_OUTLIERS),
    "hasseasonality": int(MAADS_HAS_SEASONALITY),
    "summer": MAADS_SUMMER,
    "winter": MAADS_WINTER,
    "shoulder": MAADS_SHOULDER,
    "trainingpercentage": int(MAADS_TRAINING_PERCENTAGE),
    "shuffle": int(MAADS_SHUFFLE),
    "deepanalysis": int(MAADS_DEEP_ANALYSIS),
    "timeout": int(MAADS_TRAINING_TIMEOUT_SEC),
}

r = requests.get(MAADS_TRAIN_URL, params=params, timeout=HTTP_REQUEST_TIMEOUT_SEC)
